<a href="https://colab.research.google.com/github/pluplu68/agent-lab/blob/main/docs/Transformer_Block_from_scratch_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 用 PyTorch 实现完整 Transformer Block

**目标：**从零实现一个 **Transformer Encoder Block（Post-LayerNorm 版本）**，理解 Multi-Head Attention、Feed-Forward Network、Residual Connection 和 Layer Normalization，并验证输出 Shape。

本 Notebook 可以直接在 **Jupyter / Google Colab 的 CPU Runtime** 运行，无需 GPU。

> 注意：这里是为学习而写的**简化 Encoder Block**，没有位置编码、Token Embedding 或完整训练任务；随机初始化的 Block 不会自动具备语言理解能力。

## 1.0 先理解整体数据流

假设：`batch_size=2`，`seq_len=10`，`d_model=512`，`num_heads=8`。

```text
输入 X                         (2, 10, 512)
   │
   ├─ 线性映射 W_Q、W_K、W_V    (2, 10, 512)
   ├─ 拆成 8 个 Heads           (2, 8, 10, 64)
   ├─ Q @ Kᵀ + Softmax         (2, 8, 10, 10)  ← Attention Weights
   ├─ Weights @ V              (2, 8, 10, 64)
   └─ 合并 Heads + W_O         (2, 10, 512)
            │
       残差连接 + LayerNorm
            │
       Feed-Forward Network
            │
       残差连接 + LayerNorm
            ▼
输出                            (2, 10, 512)
```

关键关系：`d_k = d_model // num_heads = 512 // 8 = 64`。

**Attention Weights 的 Shape 为 `(batch, num_heads, query_len, key_len)`，不是 `d_model`。**

In [1]:
import math
import torch
import torch.nn as nn

# 固定随机种子，方便复现实验
torch.manual_seed(42)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.14.1+cpu
CUDA available: False


## 1.1 完整代码

### A. Multi-Head Attention

**Q、K、V 的来源：**输入向量分别经过 `W_Q`、`W_K`、`W_V` 三个可训练线性层。对于 Self-Attention，通常传入同一个 `x` 作为 Q、K、V 的输入。

核心公式：

\[
\operatorname{Attention}(Q,K,V) = \operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}+M\right)V
\]

其中 `M` 表示可选的遮挡规则。这里规定：**mask 为 `1/True` 表示可见，`0/False` 表示禁止关注**。支持 Shape `(seq_q, seq_k)`、`(batch, seq_q, seq_k)` 或 `(batch, heads, seq_q, seq_k)`。

In [2]:
class MultiHeadAttention(nn.Module):
    """多头注意力层；返回输出和每个 Attention Head 的权重。"""

    def __init__(self, d_model=512, num_heads=8):
        super().__init__()
        assert d_model % num_heads == 0, "d_model 必须能被 num_heads 整除"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # 三个可训练线性映射（包含全部 Heads 的参数）
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def split_heads(self, x, batch_size):
        """(batch, seq_len, d_model) -> (batch, heads, seq_len, d_k)"""
        x = x.reshape(batch_size, -1, self.num_heads, self.d_k)
        return x.transpose(1, 2)

    def forward(self, Q, K, V, mask=None):
        batch_size = Q.shape[0]

        # Step 1: 线性变换
        Q = self.W_Q(Q)
        K = self.W_K(K)
        V = self.W_V(V)

        # Step 2: 拆分成多个 Attention Heads
        Q = self.split_heads(Q, batch_size)
        K = self.split_heads(K, batch_size)
        V = self.split_heads(V, batch_size)

        # Step 3: 计算 Scaled Dot-Product Attention Scores
        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / math.sqrt(self.d_k)

        # Step 4: 可选 Mask（修复原版 3D mask 无法正确广播到 Heads 的问题）
        if mask is not None:
            mask = mask.to(device=scores.device, dtype=torch.bool)
            if mask.ndim == 2:           # (seq_q, seq_k)
                mask = mask.unsqueeze(0).unsqueeze(0)
            elif mask.ndim == 3:         # (batch, seq_q, seq_k)
                mask = mask.unsqueeze(1)
            elif mask.ndim != 4:         # (batch, heads, seq_q, seq_k)
                raise ValueError("mask 需要是 2D、3D 或 4D Tensor")
            # 每个 Query 至少需要一个未屏蔽的 Key
            if not torch.all(mask.any(dim=-1)):
                raise ValueError("mask 不能把某个 Query 的所有 Key 都屏蔽")
            scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)

        # Step 5: Softmax，沿着 Keys 维度归一化
        attention_weights = torch.softmax(scores, dim=-1)

        # Step 6: 按权重汇总 Value，得到每个 Head 的 z
        output = torch.matmul(attention_weights, V)

        # Step 7: 合并 Heads
        output = output.transpose(1, 2).contiguous()
        output = output.reshape(batch_size, -1, self.d_model)

        # Step 8: 输出线性变换
        output = self.W_O(output)
        return output, attention_weights

### B. Feed-Forward Network (FFN)

每个 Token 的表示会**独立**经过相同的两个线性层和 ReLU：

\[
\operatorname{FFN}(x)=W_2\,\operatorname{ReLU}(W_1x+b_1)+b_2
\]

隐藏维度先从 `512` 扩展到 `2048`，再回到 `512`，因此输入和输出 Shape 相同。

In [ ]:
class FeedForward(nn.Module):
    """逐 Token 前馈网络。"""

    def __init__(self, d_model=512, d_ff=2048):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.linear2(self.relu(self.linear1(x)))

### C. 完整 Transformer Encoder Block

这个版本使用 **Post-LN**：

1. `Multi-Head Attention → Dropout → Residual Add → LayerNorm`
2. `FFN → Dropout → Residual Add → LayerNorm`

其中 Residual Connection 就是 `x + sublayer_output`。注意这里实现的是 **Encoder Block**；Decoder 使用的 Causal Mask 是可选的参数，不会自动启用。

In [ ]:
class TransformerBlock(nn.Module):
    """简化 Transformer Encoder Block（Post-LN）。"""

    def __init__(self, d_model=512, num_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Step 1: Self-Attention + Residual + LayerNorm
        attn_output, attention_weights = self.attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))

        # Step 2: FFN + Residual + LayerNorm
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x, attention_weights

## 1.2 使用示例：检查三个关键 Shapes

这里**不需要** `if __name__ == "__main__":`：在 Notebook 中直接运行代码单元即可。（如果把这段复制到 `.py` 脚本，正确写法是 `if __name__ == "__main__":`，而不是 `if name == ...`。）

In [ ]:
batch_size = 2
seq_len = 10
d_model = 512
num_heads = 8

transformer_block = TransformerBlock(d_model=d_model, num_heads=num_heads)
transformer_block.eval()  # 关闭 Dropout，方便查看稳定的结果

x = torch.randn(batch_size, seq_len, d_model)
with torch.no_grad():
    output, attention_weights = transformer_block(x)

print(f"输入形状:       {x.shape}")
print(f"输出形状:       {output.shape}")
print(f"注意力权重形状: {attention_weights.shape}")

assert output.shape == (2, 10, 512)
assert attention_weights.shape == (2, 8, 10, 10)
print("✅ Shape checks passed")

## 1.3 代码详解

### 关键点 1：`split_heads()`

原始 Tensor 的 Shape 是 `(batch, seq_len, d_model)`：

```text
(2, 10, 512)
    ↓ reshape：将 512 拆成 8 × 64
(2, 10, 8, 64)
    ↓ transpose：把 Heads 调整到 Sequence 前
(2, 8, 10, 64)
```

**没有凭空增加数据：**`8 × 64 = 512`，只是重新组织维度，让每个 Head 独立做 Attention。

In [ ]:
projected_q = transformer_block.attention.W_Q(x)
split_q = transformer_block.attention.split_heads(projected_q, batch_size)

print("W_Q(X):", tuple(projected_q.shape))
print("split_heads(W_Q(X)):", tuple(split_q.shape))
print("每个 Head 的维度 d_k:", transformer_block.attention.d_k)
assert split_q.shape == (2, 8, 10, 64)

### 关键点 2：Attention Weights 与 z

对于每个 Head：

\[
Q:(10,64),\quad K^T:(64,10)\quad\Rightarrow\quad QK^T:(10,10)
\]

这个 `10×10` 是 **10 个 Query Tokens 对 10 个 Key Tokens 的权重**；每一行经过 Softmax 后和为 1。

接着：

\[
Z = AV:\quad (10,10)\times (10,64)=(10,64)
\]

8 个 Heads 各自得到 `(10, 64)`，合并后重新变成 `(10, 512)`，再经过 `W_O`。

In [ ]:
print("一个 Head 的 Attention Matrix shape:", tuple(attention_weights[0, 0].shape))
print("第 1 条文本 / 第 1 个 Head / 第 1 个 Query 的权重前 5 项:")
print(attention_weights[0, 0, 0, :5])

row_sums = attention_weights.sum(dim=-1)
print("所有 Query 的 Attention 权重行和接近 1:",
      torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-6))
assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-6)

### 关键点 3：Residual Connection（残差连接）

\[
y=\operatorname{LayerNorm}(x+\operatorname{Dropout}(\operatorname{Attention}(x)))
\]

`+ x` 是**保留输入信息的捷径**，同时为反向传播提供更直接的梯度路径，帮助训练深层模型。后面的 FFN 也使用同样的残差结构。

**注意：**残差连接要求相加的两个 Tensor Shape 一样，所以最后需要 Output Projection `W_O` 把多个 Heads 合并到 `d_model`。

In [ ]:
attn_output, _ = transformer_block.attention(x, x, x)
print("输入 X shape:", tuple(x.shape))
print("Attention 输出 shape:", tuple(attn_output.shape))
print("残差相加后 shape:", tuple((x + attn_output).shape))
assert x.shape == attn_output.shape

### 关键点 4：Layer Normalization

`nn.LayerNorm(d_model)` 沿**最后一维**归一化，即对**每条文本每个 Token 的 `d_model` 个特征值**分别归一化，而不是跨 Batch 或跨所有 Token 一起计算。

主要作用：**改善数值稳定性和训练过程**。在默认初始参数（scale=1、bias=0）下，一个 Token 归一化后均值接近 0、方差接近 1。

In [ ]:
sample = torch.tensor([[[1.0, 2.0, 3.0, 4.0],
                        [10.0, 20.0, 30.0, 40.0]]])  # (1, 2, 4)
ln = nn.LayerNorm(4)
normalized = ln(sample)

print("输入 Shape:", tuple(sample.shape))
print("输出 Shape:", tuple(normalized.shape))
print("每个 Token 的均值:", normalized.mean(dim=-1))
print("每个 Token 的方差（population）:", normalized.var(dim=-1, unbiased=False))

### 关键点 5：Mask 的 Shape 为什么重要？

Attention Scores 的 Shape 是 `(batch, heads, query_len, key_len)`。如果传入 `(batch, query_len, key_len)` 的 Mask，就需要**在 Heads 维度插入一个 1**：

```text
Scores: (2, 8, 10, 10)
Mask:   (2, 1, 10, 10)  ← 所有 Heads 共享相同的 Mask
```

这就是本 Notebook 对原始代码所做的一个小修正。下面演示 **Causal Mask**：每个 Token 只能关注自身及之前的 Tokens。

In [ ]:
causal_mask = torch.tril(torch.ones(seq_len, seq_len, dtype=torch.bool))

with torch.no_grad():
    masked_output, masked_weights = transformer_block(x, mask=causal_mask)

print("Causal Mask（1=允许，0=屏蔽）前 5 行：")
print(causal_mask[:5, :5].int())
print("Mask 后 Attention Weights Shape:", masked_weights.shape)

# 严格上三角 = 所有未来位置，应当没有 Attention Weight
future_positions = torch.triu(
    torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1
)
assert torch.all(masked_weights[..., future_positions] == 0)
print("✅ 所有未来 Tokens 的 Attention Weights = 0")

# 也测试原教程中提到的 (batch, seq_len, seq_len) Mask
batched_mask = causal_mask.unsqueeze(0).expand(batch_size, -1, -1)
with torch.no_grad():
    _, batch_mask_weights = transformer_block(x, mask=batched_mask)
assert torch.all(batch_mask_weights[..., future_positions] == 0)
print("✅ 3D Mask (batch, seq_len, seq_len) 也能正确广播")

## 1.4 简单测试：梯度能否反向传播？

创建了模型并不等于完成训练。下面只是确认这些线性层是**可训练的**：loss 可以反向传播到参数。没有使用真实标签，也没有进行模型优化。

In [ ]:
transformer_block.train()       # 打开 Dropout，模拟训练模式
transformer_block.zero_grad()  # 清空之前可能积累的梯度

train_output, _ = transformer_block(x)
loss = train_output[:, 0, :8].sum()  # 仅用于演示的标量目标
loss.backward()

grad = transformer_block.attention.W_Q.weight.grad
print("W_Q 梯度 Shape:", tuple(grad.shape))
print("是否有有限梯度:", torch.isfinite(grad).all().item())
assert grad is not None and torch.isfinite(grad).all()

transformer_block.eval()

## 1.5 实际开发中有现成的 PyTorch 模块

刚才手动实现 Q、K、V、Softmax 和多个 Heads，是为了理解原理。实际开发中通常可以直接使用：

- `nn.MultiheadAttention`：多头注意力层。
- `nn.TransformerEncoderLayer`：封装注意力、FFN、Normalization 等的完整 Encoder Layer。
- Hugging Face `AutoModel.from_pretrained(...)`：加载现成的预训练模型。

下面仅展示 PyTorch 官方 `nn.MultiheadAttention` 的输出 Shape（默认会对 Heads 的权重求平均；这里用 `average_attn_weights=False` 保留每个 Head 的权重）。

In [ ]:
builtin_attn = nn.MultiheadAttention(
    embed_dim=512,
    num_heads=8,
    batch_first=True
)
builtin_attn.eval()

with torch.no_grad():
    builtin_output, builtin_weights = builtin_attn(
        x, x, x, average_attn_weights=False
    )

print("nn.MultiheadAttention Output:", builtin_output.shape)
print("每个 Head 的 Attention Weights:", builtin_weights.shape)
assert builtin_output.shape == (2, 10, 512)
assert builtin_weights.shape == (2, 8, 10, 10)

## 1.6 本章总结（Key Takeaways）

| Concept | 最重要的理解 |
|---|---|
| `W_Q, W_K, W_V` | 把输入分别映射为 Query、Key、Value |
| `split_heads` | `(batch, seq, 512) → (batch, 8, seq, 64)` |
| Attention Weights | `(batch, 8, seq_q, seq_k)`，给每个 Query 对各 Key 的权重 |
| `Z = softmax(QKᵀ/√d_k)V` | 对 Values 做加权求和，生成各个 Head 的新表示 |
| `W_O` | 把多个 Heads 合并后的向量映射回 `d_model` |
| Residual Connection | 加回子层输入，保留信息并辅助梯度传播 |
| LayerNorm | 在每个 Token 的 Feature Dimension 上归一化 |
| FFN | 每个 Token 独立经过相同的非线性变换 |
| Mask | 控制哪些 Key 可以被 Query 关注；Causal Mask 屏蔽未来 Tokens |

**最终 Shape（本例）**：输入 `(2,10,512)` → 输出 `(2,10,512)`，权重 `(2,8,10,10)`。
